# CASMI26: engine 2 fingerprint models, retrained with fold 0 withheld

Retrains engine 2's two FPNets (`fp_single_s2`, `fp_merged_m1`) with the original trainers (cubres/enveda-casmi-2026,
Apache 2.0), leaving out every structure of our fold-safe split (identity fold 0 + NP panel). Their scores on those
structures are then honest features for the stacked reranker. Also records which withheld structures are in MassSpecGym.

**Before you run:** GPU runtime (L4 or T4) with **High-RAM**, the `KAGGLE_API_TOKEN` secret, and allow the Google Drive prompt.
Then *Run all*. Everything lands in `MyDrive/casmi26/e2fp/`. If Colab disconnects, *Run all* again: finished steps are skipped
and a model in progress resumes from its best checkpoint.

Time: data prep about 30-40 min; single-spectrum model up to 5 h; merged model up to 7 h (the original time caps).

In [ ]:
# 1. Drive, credentials, dependencies, inputs, embedded sources
import os, sys, glob, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
def sh(cmd, check=True):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    if check and r.returncode: raise RuntimeError(cmd)
    return r
sh(f'{sys.executable} -m pip install -q -U kaggle rdkit pyarrow huggingface_hub')
OUT = '/content/drive/MyDrive/casmi26/e2fp'; os.makedirs(OUT, exist_ok=True)
R = '/content/e2'; os.makedirs(R, exist_ok=True)
SRC = '/content/e2fp'
FILES = {'prep_e2.py': '"""Build the inputs of engine 2\'s FPNet trainers (third_party/train_fp3.py, train_fp4.py) with our holdout withheld.\n\nSame layout as the legacy recipe (cubres/enveda-casmi-2026, reproduction/legacy-v2/public_source: build_fps, build_dataset,\nbuild_pool), with two changes:\n  * is_val = the structure is in the fold-safe split (identity fold 0 + NP panel), so the trainers never fit on it;\n  * the 6,930-bit selection is NOT recomputed: we use the fp_bits.npy shipped with prvsiyan/coconut-casmi26-candidates,\n    the one engine 2 uses at inference, so the new checkpoints are drop-ins for fp_single_s2 / fp_merged_m1.\nThe COCONUT part of the pool is that dataset\'s coco_fp/coco_mass/coco_meta (same bits), so no COCONUT CSV is needed.\n\nusage: prep_e2.py TRAIN_PARQUET COCO_DIR SPLIT_PARQUET OUT_DIR [--workers N]\nwrites OUT_DIR/ds/spec.npz, OUT_DIR/up_train/fp_meta.pkl, OUT_DIR/pool_{fp,mass}.npy, OUT_DIR/pool_meta.pkl\n"""\nimport argparse, os, pickle, sys, time\nfrom multiprocessing import Pool\nimport numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq\n\nsys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), \'third_party\'))\nimport fpmodel  # noqa: E402\n\nap = argparse.ArgumentParser()\nap.add_argument(\'train\'); ap.add_argument(\'coco\'); ap.add_argument(\'split\'); ap.add_argument(\'out\')\nap.add_argument(\'--workers\', type=int, default=os.cpu_count())\na = ap.parse_args()\nt0 = time.time()\nlog = lambda m: print(f\'[{time.time() - t0:6.0f}s] {m}\', flush=True)\nBITS = np.load(os.path.join(a.coco, \'fp_bits.npy\'))\nassert len(BITS) == 6930, len(BITS)\n_g = {}\n\n\ndef fp_and_mass(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import MACCSkeys, rdFingerprintGenerator\n    from rdkit.Chem.Descriptors import ExactMolWt\n    if not _g:\n        RDLogger.DisableLog(\'rdApp.*\')\n        _g[\'m2\'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n        _g[\'m3\'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\n        _g[\'rk\'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\n    m = Chem.MolFromSmiles(smi)\n    if m is None:\n        return None\n    try:\n        fp = np.concatenate([_g[\'m2\'].GetFingerprintAsNumPy(m).astype(np.uint8), _g[\'m3\'].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g[\'rk\'].GetFingerprintAsNumPy(m).astype(np.uint8), np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])\n        return np.packbits(fp[BITS]), float(ExactMolWt(m))\n    except Exception:\n        return None\n\n\ndef prep_chunk(args):\n    return [fpmodel.prep_peaks(mz, it, p) for mz, it, p in zip(*args)]\n\n\nif __name__ == \'__main__\':\n    # ---- spectra (speclib.load_flat: both peak arrays rounded to float32 first) ----\n    cols = [\'inchikey14\', \'normalized_smiles\', \'adduct\', \'precursor_mz\', \'instrument_type\', \'ionization_mode\']\n    t = pq.read_table(a.train, columns=cols + [\'ms2_mzs\', \'ms2_normalized_intensities\', \'precursor_error_ppm\', \'collision_energy_ev\'])\n    mzc = t.column(\'ms2_mzs\').combine_chunks(); itc = t.column(\'ms2_normalized_intensities\').combine_chunks()\n    off = mzc.offsets.to_numpy().astype(np.int64)\n    AMZ = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)\n    AIN = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)\n    meta = {c: (t.column(c).to_numpy(zero_copy_only=False) if pa.types.is_floating(t.column(c).type)\n                else np.asarray(t.column(c).cast(pa.string()).to_pylist(), dtype=object)) for c in cols}\n    ppm = t.column(\'precursor_error_ppm\').to_numpy(zero_copy_only=False).astype(np.float64)\n    ce = np.array([float(np.mean(v)) if v is not None and len(v) > 0 else -1.0 for v in t.column(\'collision_energy_ev\').to_pylist()], np.float32)\n    del t, mzc, itc\n    n = len(off) - 1\n    log(f\'{n:,} spectra, {len(AMZ):,} peaks\')\n\n    # ---- structures: training structures with valid fingerprints ----\n    ik, smi = meta[\'inchikey14\'], meta[\'normalized_smiles\']\n    first = pd.DataFrame({\'k\': ik, \'s\': smi}).dropna()\n    first = first[(first.k != \'\') & (first.s != \'\')].drop_duplicates(\'k\').sort_values(\'k\')\n    with Pool(a.workers) as p:\n        res = p.map(fp_and_mass, list(first.s), chunksize=500)\n    ok = [i for i, r in enumerate(res) if r is not None]\n    tr_keys = list(first.k.values[ok]); tr_smi = list(first.s.values[ok])\n    tr_fp = np.stack([res[i][0] for i in ok]); tr_mass = np.array([res[i][1] for i in ok]); del res\n    log(f\'training structures with fingerprints: {len(tr_keys):,}\')\n    os.makedirs(os.path.join(a.out, \'up_train\'), exist_ok=True)\n    pickle.dump({\'keys\': tr_keys, \'smiles\': tr_smi, \'nbits\': 6930}, open(os.path.join(a.out, \'up_train\', \'fp_meta.pkl\'), \'wb\'), protocol=4)\n\n    # ---- pool = training structures + COCONUT (legacy build_pool, COCONUT rows taken prebuilt) ----\n    cm = pickle.load(open(os.path.join(a.coco, \'coco_meta.pkl\'), \'rb\'))\n    co_keys = np.asarray(cm[\'keys\'], dtype=object); co_smi = np.asarray(cm[\'smiles\'], dtype=object)\n    co_fp = np.load(os.path.join(a.coco, \'coco_fp.npy\')); co_mass = np.load(os.path.join(a.coco, \'coco_mass.npy\'))\n    # bit-layout check: our fingerprints of a few COCONUT SMILES must equal the shipped rows\n    chk = np.random.default_rng(0).choice(len(co_keys), 200, replace=False)\n    same = [np.array_equal(fp_and_mass(co_smi[i])[0], co_fp[i, :tr_fp.shape[1]]) for i in chk if fp_and_mass(co_smi[i]) is not None]\n    log(f\'bit-layout check vs shipped COCONUT fingerprints: {np.mean(same):.3f} identical ({len(same)} checked)\')\n    assert np.mean(same) > 0.95, \'fingerprint layout differs from engine 2\'\n    new = ~pd.Series(co_keys).isin(set(tr_keys)).values\n    FP = np.concatenate([tr_fp, co_fp[new][:, :tr_fp.shape[1]]]); MASS = np.concatenate([tr_mass, co_mass[new]])\n    KEYS = np.concatenate([np.array(tr_keys, dtype=object), co_keys[new]]); SMI = np.concatenate([np.array(tr_smi, dtype=object), co_smi[new]])\n    SRC = np.concatenate([np.zeros(len(tr_keys), np.int8), np.ones(int(new.sum()), np.int8)])\n    good = np.isfinite(MASS) & (MASS >= 80) & (MASS <= 1400)\n    o = np.flatnonzero(good)[np.argsort(MASS[good], kind=\'mergesort\')]\n    np.save(os.path.join(a.out, \'pool_fp.npy\'), FP[o]); np.save(os.path.join(a.out, \'pool_mass.npy\'), MASS[o])\n    pickle.dump({\'keys\': KEYS[o], \'smiles\': SMI[o], \'src\': SRC[o], \'npl\': np.full(len(o), np.nan), \'nbits\': 6930},\n                open(os.path.join(a.out, \'pool_meta.pkl\'), \'wb\'), protocol=4)\n    log(f\'pool {len(o):,} structures ({int(new.sum()):,} from COCONUT)\')\n    del FP, co_fp\n\n    # ---- usable spectra, holdout flag, peak preprocessing (legacy build_dataset) ----\n    key2ix = {k: i for i, k in enumerate(tr_keys)}\n    six = np.array([key2ix.get(k, -1) for k in ik], np.int32)\n    npk = np.diff(off)\n    use = np.isfinite(meta[\'precursor_mz\']) & (npk >= 3) & ~(np.isfinite(ppm) & (np.abs(ppm) > 20)) & (six >= 0)\n    idx = np.flatnonzero(use)\n    hold = set(pd.read_parquet(a.split).inchikey14)\n    is_val = np.array([ik[i] in hold for i in idx])\n    log(f\'usable spectra {len(idx):,}; withheld {is_val.sum():,} spectra of {len({ik[i] for i in idx[is_val]}):,} \'\n        f\'structures (split lists {len(hold):,}); training {(~is_val).sum():,}\')\n    CH = 20000\n    chunks = [([AMZ[off[i]:off[i + 1]] for i in idx[s:s + CH]], [AIN[off[i]:off[i + 1]] for i in idx[s:s + CH]],\n               [meta[\'precursor_mz\'][i] for i in idx[s:s + CH]]) for s in range(0, len(idx), CH)]\n    del AMZ, AIN\n    res = []\n    with Pool(a.workers) as p:\n        for j, r in enumerate(p.imap(prep_chunk, chunks)):\n            res.extend(r)\n            if j % 20 == 0:\n                log(f\'  peaks chunk {j}/{len(chunks)}\')\n    lens = np.array([len(x) for x, _ in res], np.int64)\n    noff = np.zeros(len(res) + 1, np.int64); np.cumsum(lens, out=noff[1:])\n    ad = np.array([fpmodel.ADDUCT_IX.get(x, fpmodel.ADDUCT_IX[\'<unk>\']) for x in meta[\'adduct\'][idx]], np.int16)\n    ins = np.array([fpmodel.instr_family(s) for s in meta[\'instrument_type\'][idx]], np.int8)\n    mode = np.array([1.0 if str(m) == \'positive\' else -1.0 for m in meta[\'ionization_mode\'][idx]], np.float32)\n    os.makedirs(os.path.join(a.out, \'ds\'), exist_ok=True)\n    np.savez(os.path.join(a.out, \'ds\', \'spec.npz\'), off=noff, mz=np.concatenate([x for x, _ in res]).astype(np.float32),\n             it=np.concatenate([y for _, y in res]).astype(np.float32), prec=meta[\'precursor_mz\'][idx].astype(np.float32), ad=ad,\n             ins=ins, ce=ce[idx], mode=mode, six=six[idx], is_val=is_val, src=idx.astype(np.int64))\n    log(\'saved \' + a.out)\n', 'msg_overlap.py': '"""Which of our withheld structures are in MassSpecGym (the training data of ICEBERG and GLACIER)?\n\nICEBERG/GLACIER scores on a withheld structure that MassSpecGym trained on are optimistic, so reranker rows for those\nstructures get a flag. Reads only the inchikey and fold columns of the MassSpecGym table on Hugging Face.\nusage: msg_overlap.py SPLIT_PARQUET OUT_JSON\n"""\nimport json, sys\nimport pandas as pd\nfrom huggingface_hub import hf_hub_download, list_repo_files\n\nsplit, out = sys.argv[1:3]\nrepo = \'roman-bushuiev/MassSpecGym\'\ntsv = sorted((f for f in list_repo_files(repo, repo_type=\'dataset\') if f.endswith(\'MassSpecGym.tsv\')), key=len)[0]\nm = pd.read_csv(hf_hub_download(repo, tsv, repo_type=\'dataset\'), sep=\'\\t\', usecols=[\'inchikey\', \'fold\'])\nm[\'ik14\'] = m.inchikey.str[:14]\nsp = pd.read_parquet(split)\nres = {\'source\': f\'{repo}/{tsv}\', \'folds\': {}}\nfor f, g in m.groupby(\'fold\'):\n    keys = set(g.ik14)\n    res[\'folds\'][f] = sorted(keys)\n    print(f\'MassSpecGym {f}: {len(keys):,} structures; in our withheld split: \'\n          + \', \'.join(f\'{k} {sp[sp.fold == k].inchikey14.isin(keys).sum():,}/{(sp.fold == k).sum():,}\' for k in sorted(sp.fold.unique())), flush=True)\njson.dump(res, open(out, \'w\'))\nprint(\'wrote\', out)\n', 'third_party/fpmodel.py': '"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""\nimport numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math\n\nMAX_PEAKS = 128\nADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",\n               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",\n               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",\n               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]\nADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}\nINSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]\nINSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}\n\ndef instr_family(s):\n    if s is None: return 4\n    t = str(s).lower()\n    if \'timstof\' in t: return 0\n    if \'orbitrap\' in t or \'qft\' in t or \'ftms\' in t or \'hybrid ft\' in t or \'itft\' in t or \'exactive\' in t: return 1\n    if \'tof\' in t: return 2\n    if \'trap\' in t or \'qq\' in t: return 3\n    return 4\n\ndef prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\n    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\n    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)\n    keep = (mz <= prec_mz + 1.5)\n    mz, it = mz[keep], it[keep]\n    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)\n    mx = it.max()\n    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)\n    keep = it >= floor*mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N\n        order = np.argsort(-it)\n        bucket = (mz//win).astype(np.int64)\n        cnt = {}; sel=[]\n        for i in order:\n            b = bucket[i]; c = cnt.get(b,0)\n            if c < per_win: cnt[b]=c+1; sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\n            need = max_peaks-len(sel)\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    v = np.sqrt(it/it.max())\n    return mz.astype(np.float32), v.astype(np.float32)\n\nclass SinEmb(nn.Module):\n    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\n        super().__init__()\n        n = dim//2\n        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)\n        self.register_buffer(\'inv\', (2*math.pi)/wav)\n    def forward(self, x):                      # x: (...,)\n        a = x.unsqueeze(-1) * self.inv\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\nclass Block(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__(); self.h=h\n        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)\n        self.n2=nn.LayerNorm(d)\n        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))\n        self.drop=nn.Dropout(drop)\n    def forward(self, x, pad):\n        B,N,D=x.shape; y=self.n1(x)\n        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)\n        m = (~pad)[:,None,None,:]                       # True = attend\n        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)\n        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))\n        return x + self.drop(self.ff(self.n2(x)))\n\nclass FPNet(nn.Module):\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\n        super().__init__()\n        self.d=d\n        self.mz_emb  = SinEmb(d)\n        self.nl_emb  = SinEmb(d)\n        self.pk = nn.Linear(2*d+1, d)\n        self.prec_emb = SinEmb(d)\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\n        self.gl = nn.Linear(d+3, d)\n        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n        B,N = mz.shape\n        nl = (prec[:,None] - mz).clamp(min=0)\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n        g = self.gl(torch.cat([self.prec_emb(prec),\n                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)\n        x = torch.cat([g.unsqueeze(1), p], 1)\n        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)\n        for b in self.blocks: x = b(x, pad)\n        x = self.norm(x)\n        cls = x[:,0]\n        msk = (~pad[:,1:]).float().unsqueeze(-1)\n        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)\n        return self.head(torch.cat([cls, mean], -1))\n', 'third_party/train_fp3.py': '"""Spectrum->fingerprint training with BCE + hard-negative contrastive ranking loss.\n   Ranking score is f_cand . z (exact Bayes log-likelihood up to a candidate-independent constant)."""\nimport numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, pickle, os, math, argparse\nimport fpmodel\n\ndef dev_of():\n    if torch.cuda.is_available(): return \'cuda\'\n    if torch.backends.mps.is_available(): return \'mps\'\n    return \'cpu\'\n\nclass Data:\n    def __init__(self, spec=\'ds/spec.npz\', fpdir=\'up_train\', pooldir=\'.\', device=\'cpu\', pool_on_gpu=True):\n        z=np.load(spec)\n        self.off=z[\'off\']; self.mz=z[\'mz\']; self.it=z[\'it\']; self.prec=z[\'prec\']\n        self.ad=z[\'ad\'].astype(np.int64); self.ins=z[\'ins\'].astype(np.int64)\n        self.ce=z[\'ce\']; self.mode=z[\'mode\']; self.six=z[\'six\']; self.is_val=z[\'is_val\']\n        m=pickle.load(open(f\'{fpdir}/fp_meta.pkl\',\'rb\')); self.nbits=m[\'nbits\']; self.skeys=m[\'keys\']\n        pm=pickle.load(open(f\'{pooldir}/pool_meta.pkl\',\'rb\'))\n        self.pmass=np.load(f\'{pooldir}/pool_mass.npy\')\n        PFP=np.unpackbits(np.load(f\'{pooldir}/pool_fp.npy\'),axis=1)[:, :self.nbits]\n        self.pkeys=pm[\'keys\']\n        k2p={k:i for i,k in enumerate(self.pkeys)}\n        self.s2p=np.array([k2p.get(k,-1) for k in self.skeys], dtype=np.int64)\n        self.device=device\n        if pool_on_gpu and device==\'cuda\':\n            self.PFP=torch.from_numpy(np.ascontiguousarray(PFP)).to(device)\n        else:\n            self.PFP=torch.from_numpy(np.ascontiguousarray(PFP))\n        self.tr=np.where(~self.is_val)[0]; self.va=np.where(self.is_val)[0]\n        keep = self.s2p[self.six[self.tr]]>=0\n        self.tr = self.tr[keep]\n        print(f\'nbits={self.nbits} pool={len(self.pmass)} train={len(self.tr)} val={len(self.va)}\',flush=True)\n\n    def sample_negs(self, pos_pidx, K, ppm=10.0, rng=None):\n        m=self.pmass[pos_pidx]; tol=m*ppm/1e6\n        lo=np.searchsorted(self.pmass, m-tol,\'left\'); hi=np.searchsorted(self.pmass, m+tol,\'right\')\n        out=np.empty((len(pos_pidx),K), dtype=np.int64)\n        for r,(a,b,p) in enumerate(zip(lo,hi,pos_pidx)):\n            n=b-a\n            if n<=1: out[r]=rng.integers(0,len(self.pmass),K)   # no isomers -> random decoys\n            else:\n                c=rng.integers(a,b,K)\n                bad=(c==p)\n                if bad.any(): c[bad]=np.where(c[bad]+1<b, c[bad]+1, a)\n                out[r]=c\n        return out\n\n    def batch(self, ids, K=0, rng=None, ppm=10.0, aug=False):\n        B=len(ids); L=fpmodel.MAX_PEAKS\n        lens=np.minimum(self.off[ids+1]-self.off[ids], L); N=max(int(lens.max()),1)\n        mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)\n        for r,(i,l) in enumerate(zip(ids,lens)):\n            a=self.off[i]; mz[r,:l]=self.mz[a:a+l]; it[r,:l]=self.it[a:a+l]; pad[r,:l]=False\n        if aug and rng is not None:\n            # spectra are noisy measurements; jitter them so the model cannot memorise\n            # exact peak patterns of the 276k training structures (the earlier run\n            # overfit hard: val hardneg-top1 peaked at 0.46 then decayed to 0.13).\n            keep = rng.random((B,N)) > rng.uniform(0.0, 0.30, size=(B,1))   # peak dropout\n            pad = pad | (~keep)\n            it = it * np.exp(rng.normal(0.0, 0.25, size=(B,N))).astype(np.float32)  # intensity jitter\n            mz = mz * (1.0 + rng.normal(0.0, 5e-6, size=(B,N))).astype(np.float32)  # m/z jitter (~5 ppm)\n            allpad = pad.all(1)\n            if allpad.any(): pad[allpad,0]=False\n        d=self.device; T=lambda x: torch.as_tensor(x,device=d)\n        ce=np.where(self.ce[ids]<0,25.0,self.ce[ids]).astype(np.float32)\n        inp=(T(mz),T(it),T(pad),T(self.prec[ids]),T(self.ad[ids]),T(self.ins[ids]),T(ce),T(self.mode[ids]))\n        pos=self.s2p[self.six[ids]]\n        cand=None\n        if K>0:\n            negs=self.sample_negs(pos,K,ppm,rng)\n            cand=np.concatenate([pos[:,None],negs],1)         # (B,K+1), col 0 = positive\n            cand=torch.as_tensor(cand, device=self.PFP.device)\n        ypos=self.PFP[torch.as_tensor(pos, device=self.PFP.device)].to(d).float()\n        return inp, ypos, cand\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--steps\',type=int,default=80000); ap.add_argument(\'--bs\',type=int,default=256)\n    ap.add_argument(\'--lr\',type=float,default=3e-4); ap.add_argument(\'--d\',type=int,default=512)\n    ap.add_argument(\'--layers\',type=int,default=6); ap.add_argument(\'--K\',type=int,default=63)\n    ap.add_argument(\'--lam\',type=float,default=1.0); ap.add_argument(\'--warm\',type=int,default=2000)\n    ap.add_argument(\'--out\',default=\'fp_model.pt\'); ap.add_argument(\'--resume\',default=\'\')\n    ap.add_argument(\'--val_every\',type=int,default=2000); ap.add_argument(\'--max_minutes\',type=float,default=1e9)\n    ap.add_argument(\'--seed\',type=int,default=7)\n    ap.add_argument(\'--spec\',default=\'ds/spec.npz\'); ap.add_argument(\'--fpdir\',default=\'up_train\'); ap.add_argument(\'--pooldir\',default=\'.\')\n    a=ap.parse_args()\n    dev=dev_of(); print(\'device\',dev,flush=True)\n    D=Data(a.spec,a.fpdir,a.pooldir,device=dev)\n    model=fpmodel.FPNet(D.nbits,d=a.d,layers=a.layers).to(dev)\n    print(\'params %.1fM\'%(sum(p.numel() for p in model.parameters())/1e6),flush=True)\n    torch.manual_seed(a.seed)\n    opt=torch.optim.AdamW(model.parameters(),lr=a.lr,weight_decay=0.01,betas=(0.9,0.98))\n    scaler=torch.amp.GradScaler(\'cuda\') if dev==\'cuda\' else None\n    start=0\n    if a.resume and os.path.exists(a.resume):\n        ck=torch.load(a.resume,map_location=dev); model.load_state_dict(ck[\'model\'])\n        try: opt.load_state_dict(ck[\'opt\'])\n        except Exception: pass\n        start=ck[\'step\']; print(\'resume\',start,flush=True)\n    def lr_at(s):\n        if s<a.warm: return a.lr*s/max(1,a.warm)\n        p=(s-a.warm)/max(1,a.steps-a.warm); return a.lr*(0.02+0.98*0.5*(1+math.cos(math.pi*min(p,1.0))))\n    rng=np.random.default_rng(a.seed)\n    t0=time.time(); rb=rc=racc=0.0; nr=0\n    best_val=-1.0   # keep the checkpoint that generalises best, not the last one\n    def save(step):\n        torch.save({\'model\':model.state_dict(),\'opt\':opt.state_dict(),\'step\':step,\n                    \'nbits\':D.nbits,\'d\':a.d,\'layers\':a.layers}, a.out)\n    for step in range(start,a.steps):\n        for g in opt.param_groups: g[\'lr\']=lr_at(step)\n        ids=rng.choice(D.tr,size=a.bs,replace=False)\n        inp,ypos,cand = D.batch(ids,K=a.K,rng=rng,aug=True)\n        opt.zero_grad(set_to_none=True)\n        ctx = torch.autocast(\'cuda\',dtype=torch.float16) if dev==\'cuda\' else torch.autocast(\'mps\',dtype=torch.float16) if dev==\'mps\' else torch.autocast(\'cpu\',enabled=False)\n        with ctx:\n            z=model(*inp)\n        z=z.float()\n        lb=F.binary_cross_entropy_with_logits(z,ypos)\n        FPc=D.PFP[cand].to(z.device).float()                     # (B,K+1,nbits)\n        raw=torch.bmm(FPc, z.unsqueeze(-1)).squeeze(-1)          # (B,K+1) = f.z  (exact Bayes LL up to const)\n        # centre within the example: subtracting (mean_c f_c).z is a per-example constant,\n        # so the ranking is untouched, but the magnitudes stay small enough for a stable softmax.\n        sc=(raw - raw.mean(dim=1, keepdim=True)) / math.sqrt(D.nbits)\n        tgt=torch.zeros(len(ids),dtype=torch.long,device=z.device)\n        lc=F.cross_entropy(sc,tgt)\n        loss=lb+a.lam*lc\n        if not torch.isfinite(loss):\n            print(f\'  non-finite loss at step {step}; skipping\', flush=True)\n            opt.zero_grad(set_to_none=True); continue\n        if scaler is not None:\n            scaler.scale(loss).backward(); scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update()\n        else:\n            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()\n        rb+=lb.item(); rc+=lc.item(); racc+=(sc.argmax(1)==0).float().mean().item(); nr+=1\n        if step%200==0:\n            el=time.time()-t0\n            print(f\'step {step} bce {rb/max(nr,1):.4f} ctr {rc/max(nr,1):.4f} top1 {racc/max(nr,1):.3f} lr {lr_at(step):.2e} {el:.0f}s {(step-start+1)*a.bs/max(el,1):.0f} spec/s\',flush=True)\n            rb=rc=racc=0.0; nr=0\n        if (step+1)%a.val_every==0:\n            model.eval(); vb=[];vacc=[]\n            with torch.no_grad():\n                vids=rng.choice(D.va,size=min(2048,len(D.va)),replace=False)\n                for s in range(0,len(vids),128):\n                    ii=vids[s:s+128]\n                    pos=D.s2p[D.six[ii]]\n                    if (pos<0).any(): ii=ii[pos>=0]\n                    if len(ii)==0: continue\n                    inp,ypos,cand=D.batch(ii,K=a.K,rng=rng)\n                    z=model(*inp).float()\n                    vb.append(F.binary_cross_entropy_with_logits(z,ypos).item())\n                    FPc=D.PFP[cand].to(z.device).float()\n                    raw=torch.bmm(FPc,z.unsqueeze(-1)).squeeze(-1)\n                    sc=(raw-raw.mean(dim=1,keepdim=True))/math.sqrt(D.nbits)\n                    vacc.append((sc.argmax(1)==0).float().mean().item())\n            va=float(np.mean(vacc))\n            print(f\'  VAL step {step} bce {np.mean(vb):.4f} hardneg-top1 {va:.3f}\'\n                  + (\'  <- best\' if va>best_val else \'\'),flush=True)\n            model.train()\n            if va>best_val:\n                best_val=va; save(step+1)          # only overwrite when validation improves\n        if (time.time()-t0)/60>a.max_minutes:\n            print(\'time budget reached; keeping best-val checkpoint\',flush=True); break\n    print(\'done. best hardneg-top1 = %.3f\'%best_val,flush=True)\n\nif __name__==\'__main__\': main()\n', 'third_party/train_fp4.py': '"""Spectrum->fingerprint training with BCE + hard-negative contrastive ranking loss.\n   Ranking score is f_cand . z (exact Bayes log-likelihood up to a candidate-independent constant)."""\nimport numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, pickle, os, math, argparse\nimport fpmodel\n\ndef dev_of():\n    if torch.cuda.is_available(): return \'cuda\'\n    if torch.backends.mps.is_available(): return \'mps\'\n    return \'cpu\'\n\nclass Data:\n    def __init__(self, spec=\'ds/spec.npz\', fpdir=\'up_train\', pooldir=\'.\', device=\'cpu\', pool_on_gpu=True):\n        z=np.load(spec)\n        self.off=z[\'off\']; self.mz=z[\'mz\']; self.it=z[\'it\']; self.prec=z[\'prec\']\n        self.ad=z[\'ad\'].astype(np.int64); self.ins=z[\'ins\'].astype(np.int64)\n        self.ce=z[\'ce\']; self.mode=z[\'mode\']; self.six=z[\'six\']; self.is_val=z[\'is_val\']\n        m=pickle.load(open(f\'{fpdir}/fp_meta.pkl\',\'rb\')); self.nbits=m[\'nbits\']; self.skeys=m[\'keys\']\n        pm=pickle.load(open(f\'{pooldir}/pool_meta.pkl\',\'rb\'))\n        self.pmass=np.load(f\'{pooldir}/pool_mass.npy\')\n        PFP=np.unpackbits(np.load(f\'{pooldir}/pool_fp.npy\'),axis=1)[:, :self.nbits]\n        self.pkeys=pm[\'keys\']\n        k2p={k:i for i,k in enumerate(self.pkeys)}\n        self.s2p=np.array([k2p.get(k,-1) for k in self.skeys], dtype=np.int64)\n        self.device=device\n        if pool_on_gpu and device==\'cuda\':\n            self.PFP=torch.from_numpy(np.ascontiguousarray(PFP)).to(device)\n        else:\n            self.PFP=torch.from_numpy(np.ascontiguousarray(PFP))\n        # group spectra by structure so training inputs can be MERGED the way test molecules are:\n        # the hidden test gives 1-16 spectra per molecule (median 3), but training on single\n        # spectra creates a train/test mismatch.\n        order=np.argsort(self.six, kind=\'mergesort\')\n        s_sorted=self.six[order]\n        bounds=np.searchsorted(s_sorted, np.arange(s_sorted.max()+2))\n        self._grp_order=order; self._grp_bounds=bounds\n        self.tr=np.where(~self.is_val)[0]; self.va=np.where(self.is_val)[0]\n        keep = self.s2p[self.six[self.tr]]>=0\n        self.tr = self.tr[keep]\n        print(f\'nbits={self.nbits} pool={len(self.pmass)} train={len(self.tr)} val={len(self.va)}\',flush=True)\n\n    def sample_negs(self, pos_pidx, K, ppm=10.0, rng=None):\n        m=self.pmass[pos_pidx]; tol=m*ppm/1e6\n        lo=np.searchsorted(self.pmass, m-tol,\'left\'); hi=np.searchsorted(self.pmass, m+tol,\'right\')\n        out=np.empty((len(pos_pidx),K), dtype=np.int64)\n        for r,(a,b,p) in enumerate(zip(lo,hi,pos_pidx)):\n            n=b-a\n            if n<=1: out[r]=rng.integers(0,len(self.pmass),K)   # no isomers -> random decoys\n            else:\n                c=rng.integers(a,b,K)\n                bad=(c==p)\n                if bad.any(): c[bad]=np.where(c[bad]+1<b, c[bad]+1, a)\n                out[r]=c\n        return out\n\n    def peers(self, i, rng, kmax=4):\n        """Other spectra of the same structure (for merge augmentation)."""\n        s=self.six[i]\n        a,b=self._grp_bounds[s], self._grp_bounds[s+1]\n        if b-a<=1: return [i]\n        cand=self._grp_order[a:b]\n        k=int(rng.integers(2, min(kmax, b-a)+1))\n        pick=rng.choice(cand, size=min(k,len(cand)), replace=False)\n        if i not in pick: pick=np.concatenate([[i], pick[:-1]])\n        return list(pick)\n\n    def _merged(self, idxs, maxlen):\n        mzs=[]; its=[]\n        for j in idxs:\n            a,b=self.off[j], min(self.off[j]+maxlen, self.off[j+1])\n            mzs.append(self.mz[a:b]); its.append(self.it[a:b])\n        mz=np.concatenate(mzs); it=np.concatenate(its)\n        o=np.argsort(mz); mz,it=mz[o],it[o]\n        keep=np.ones(len(mz),bool)\n        for j in range(1,len(mz)):\n            if mz[j]-mz[j-1] < 0.005:\n                if it[j]>=it[j-1]: keep[j-1]=False\n                else: keep[j]=False\n        mz,it=mz[keep],it[keep]\n        if len(mz)>maxlen:\n            top=np.argsort(-it)[:maxlen]; top.sort(); mz,it=mz[top],it[top]\n        return mz,it\n\n    def batch(self, ids, K=0, rng=None, ppm=10.0, aug=False, merge_p=0.0):\n        B=len(ids); L=fpmodel.MAX_PEAKS\n        lens=np.minimum(self.off[ids+1]-self.off[ids], L); N=max(int(lens.max()),1)\n        mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)\n        for r,(i,l) in enumerate(zip(ids,lens)):\n            a=self.off[i]; mz[r,:l]=self.mz[a:a+l]; it[r,:l]=self.it[a:a+l]; pad[r,:l]=False\n        if merge_p>0 and rng is not None:\n            for r,i in enumerate(ids):\n                if rng.random()>=merge_p: continue\n                pk=self.peers(i,rng)\n                if len(pk)<2: continue\n                m2,i2=self._merged(pk, N)\n                n2=len(m2)\n                mz[r,:]=0; it[r,:]=0; pad[r,:]=True\n                mz[r,:n2]=m2; it[r,:n2]=i2; pad[r,:n2]=False\n        if aug and rng is not None:\n            # spectra are noisy measurements; jitter them so the model cannot memorise\n            # exact peak patterns of the 276k training structures (the earlier run\n            # overfit hard: val hardneg-top1 peaked at 0.46 then decayed to 0.13).\n            keep = rng.random((B,N)) > rng.uniform(0.0, 0.30, size=(B,1))   # peak dropout\n            pad = pad | (~keep)\n            it = it * np.exp(rng.normal(0.0, 0.25, size=(B,N))).astype(np.float32)  # intensity jitter\n            mz = mz * (1.0 + rng.normal(0.0, 5e-6, size=(B,N))).astype(np.float32)  # m/z jitter (~5 ppm)\n            allpad = pad.all(1)\n            if allpad.any(): pad[allpad,0]=False\n        d=self.device; T=lambda x: torch.as_tensor(x,device=d)\n        ce=np.where(self.ce[ids]<0,25.0,self.ce[ids]).astype(np.float32)\n        inp=(T(mz),T(it),T(pad),T(self.prec[ids]),T(self.ad[ids]),T(self.ins[ids]),T(ce),T(self.mode[ids]))\n        pos=self.s2p[self.six[ids]]\n        cand=None\n        if K>0:\n            negs=self.sample_negs(pos,K,ppm,rng)\n            cand=np.concatenate([pos[:,None],negs],1)         # (B,K+1), col 0 = positive\n            cand=torch.as_tensor(cand, device=self.PFP.device)\n        ypos=self.PFP[torch.as_tensor(pos, device=self.PFP.device)].to(d).float()\n        return inp, ypos, cand\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--steps\',type=int,default=80000); ap.add_argument(\'--bs\',type=int,default=256)\n    ap.add_argument(\'--lr\',type=float,default=3e-4); ap.add_argument(\'--d\',type=int,default=512)\n    ap.add_argument(\'--layers\',type=int,default=6); ap.add_argument(\'--K\',type=int,default=63)\n    ap.add_argument(\'--lam\',type=float,default=1.0); ap.add_argument(\'--warm\',type=int,default=2000)\n    ap.add_argument(\'--out\',default=\'fp_model.pt\'); ap.add_argument(\'--resume\',default=\'\')\n    ap.add_argument(\'--val_every\',type=int,default=2000); ap.add_argument(\'--max_minutes\',type=float,default=1e9)\n    ap.add_argument(\'--seed\',type=int,default=7)\n    ap.add_argument(\'--merge_p\',type=float,default=0.6)\n    ap.add_argument(\'--spec\',default=\'ds/spec.npz\'); ap.add_argument(\'--fpdir\',default=\'up_train\'); ap.add_argument(\'--pooldir\',default=\'.\')\n    a=ap.parse_args()\n    dev=dev_of(); print(\'device\',dev,flush=True)\n    D=Data(a.spec,a.fpdir,a.pooldir,device=dev)\n    model=fpmodel.FPNet(D.nbits,d=a.d,layers=a.layers).to(dev)\n    print(\'params %.1fM\'%(sum(p.numel() for p in model.parameters())/1e6),flush=True)\n    torch.manual_seed(a.seed)\n    opt=torch.optim.AdamW(model.parameters(),lr=a.lr,weight_decay=0.01,betas=(0.9,0.98))\n    scaler=torch.amp.GradScaler(\'cuda\') if dev==\'cuda\' else None\n    start=0\n    if a.resume and os.path.exists(a.resume):\n        ck=torch.load(a.resume,map_location=dev); model.load_state_dict(ck[\'model\'])\n        try: opt.load_state_dict(ck[\'opt\'])\n        except Exception: pass\n        start=ck[\'step\']; print(\'resume\',start,flush=True)\n    def lr_at(s):\n        if s<a.warm: return a.lr*s/max(1,a.warm)\n        p=(s-a.warm)/max(1,a.steps-a.warm); return a.lr*(0.02+0.98*0.5*(1+math.cos(math.pi*min(p,1.0))))\n    rng=np.random.default_rng(a.seed)\n    t0=time.time(); rb=rc=racc=0.0; nr=0\n    best_val=-1.0   # keep the checkpoint that generalises best, not the last one\n    def save(step):\n        torch.save({\'model\':model.state_dict(),\'opt\':opt.state_dict(),\'step\':step,\n                    \'nbits\':D.nbits,\'d\':a.d,\'layers\':a.layers}, a.out)\n    for step in range(start,a.steps):\n        for g in opt.param_groups: g[\'lr\']=lr_at(step)\n        ids=rng.choice(D.tr,size=a.bs,replace=False)\n        inp,ypos,cand = D.batch(ids,K=a.K,rng=rng,aug=True,merge_p=a.merge_p)\n        opt.zero_grad(set_to_none=True)\n        ctx = torch.autocast(\'cuda\',dtype=torch.float16) if dev==\'cuda\' else torch.autocast(\'mps\',dtype=torch.float16) if dev==\'mps\' else torch.autocast(\'cpu\',enabled=False)\n        with ctx:\n            z=model(*inp)\n        z=z.float()\n        lb=F.binary_cross_entropy_with_logits(z,ypos)\n        FPc=D.PFP[cand].to(z.device).float()                     # (B,K+1,nbits)\n        raw=torch.bmm(FPc, z.unsqueeze(-1)).squeeze(-1)          # (B,K+1) = f.z  (exact Bayes LL up to const)\n        # centre within the example: subtracting (mean_c f_c).z is a per-example constant,\n        # so the ranking is untouched, but the magnitudes stay small enough for a stable softmax.\n        sc=(raw - raw.mean(dim=1, keepdim=True)) / math.sqrt(D.nbits)\n        tgt=torch.zeros(len(ids),dtype=torch.long,device=z.device)\n        lc=F.cross_entropy(sc,tgt)\n        loss=lb+a.lam*lc\n        if not torch.isfinite(loss):\n            print(f\'  non-finite loss at step {step}; skipping\', flush=True)\n            opt.zero_grad(set_to_none=True); continue\n        if scaler is not None:\n            scaler.scale(loss).backward(); scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update()\n        else:\n            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()\n        rb+=lb.item(); rc+=lc.item(); racc+=(sc.argmax(1)==0).float().mean().item(); nr+=1\n        if step%200==0:\n            el=time.time()-t0\n            print(f\'step {step} bce {rb/max(nr,1):.4f} ctr {rc/max(nr,1):.4f} top1 {racc/max(nr,1):.3f} lr {lr_at(step):.2e} {el:.0f}s {(step-start+1)*a.bs/max(el,1):.0f} spec/s\',flush=True)\n            rb=rc=racc=0.0; nr=0\n        if (step+1)%a.val_every==0:\n            model.eval(); vb=[];vacc=[]\n            with torch.no_grad():\n                vids=rng.choice(D.va,size=min(2048,len(D.va)),replace=False)\n                for s in range(0,len(vids),128):\n                    ii=vids[s:s+128]\n                    pos=D.s2p[D.six[ii]]\n                    if (pos<0).any(): ii=ii[pos>=0]\n                    if len(ii)==0: continue\n                    inp,ypos,cand=D.batch(ii,K=a.K,rng=rng,merge_p=a.merge_p)\n                    z=model(*inp).float()\n                    vb.append(F.binary_cross_entropy_with_logits(z,ypos).item())\n                    FPc=D.PFP[cand].to(z.device).float()\n                    raw=torch.bmm(FPc,z.unsqueeze(-1)).squeeze(-1)\n                    sc=(raw-raw.mean(dim=1,keepdim=True))/math.sqrt(D.nbits)\n                    vacc.append((sc.argmax(1)==0).float().mean().item())\n            va=float(np.mean(vacc))\n            print(f\'  VAL step {step} bce {np.mean(vb):.4f} hardneg-top1 {va:.3f}\'\n                  + (\'  <- best\' if va>best_val else \'\'),flush=True)\n            model.train()\n            if va>best_val:\n                best_val=va; save(step+1)          # only overwrite when validation improves\n        if (time.time()-t0)/60>a.max_minutes:\n            print(\'time budget reached; keeping best-val checkpoint\',flush=True); break\n    print(\'done. best hardneg-top1 = %.3f\'%best_val,flush=True)\n\nif __name__==\'__main__\': main()\n', 'third_party/LICENSE-SOURCE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'third_party/NOTICE': 'CASMI legacy-v2 original and reconstructed campaign source\nCopyright 2026 cubres\n\nThe owner makes ONLY the following 29 source-file paths, relative to this\nreproduction/legacy-v2 directory, available under the Apache License, Version 2.0.\nThe complete license text is in LICENSE-SOURCE. Exact reviewed source hashes,\nheader findings and provenance categories are recorded in SOURCE_LICENSE_AUDIT.json.\n\n  public_source/build_coconut.py\n  public_source/build_dataset.py\n  public_source/build_fps.py\n  public_source/build_pool.py\n  public_source/demo_preparation_synthetic.py\n  public_source/fpmodel.py\n  public_source/prepare_legacy_exports.py\n  public_source/speclib.py\n  public_source/train_fp.py\n  full_fingerprint_lineage/build_clean_full_fingerprints.py\n  checkpoint_lineage/deterministic_training_launcher.py\n  checkpoint_lineage/original_sources/fpmodel.py\n  checkpoint_lineage/original_sources/train_fp3.py\n  checkpoint_lineage/original_sources/train_fp4.py\n  ranker_lineage/original_sources/build_dataset.py\n  ranker_lineage/original_sources/build_fps.py\n  ranker_lineage/original_sources/casmi_core.py\n  ranker_lineage/original_sources/fpmodel.py\n  ranker_lineage/original_sources/hard_logits_combo.py\n  ranker_lineage/original_sources/hard_logits_m1.py\n  ranker_lineage/original_sources/make_ranker3.py\n  ranker_lineage/original_sources/model_logits_hard.py\n  ranker_lineage/original_sources/rankfeat.py\n  ranker_lineage/original_sources/score_model.py\n  ranker_lineage/original_sources/train_fp.py\n  ranker_lineage/original_sources/train_fp3.py\n  ranker_lineage/ORIGINAL_LOCAL_RANKER_FIT_EXCERPT.txt\n  ranker_lineage/original_sources/engine.py\n  ranker_lineage/original_sources/speclib.py\n\nThe original file bytes, source headers, method descriptions, historical comments\nand any existing attribution notices remain unchanged. The FPNet header\'s\n"CSI:FingerID-style" description and its m/z embedding docstring crediting\n"Voronov et al." are retained as method context. These credits do not imply that\nthird-party implementation code is included. This grant does not grant rights\nto CSI:FingerID, SIRIUS, Voronov et al. implementations, or other third-party code,\ndata, dependency, patent or trademark beyond rights actually licensable under\nApache License 2.0 by the contributing copyright owner.\n\nThis notice does not license external dependencies or inputs, candidate structures,\nspectra, role assignments, model weights, caches, third-party notebooks, other\nrepository files, or the thirteen unlisted documentation/metadata/dependency-list\nfiles from the reviewed legacy release. Imported library implementations retain\ntheir own licenses. No repository-wide or upstream-data license is inferred.\n\nThree listed sources are new campaign reconstruction/demo/recipe code, not\nrecovered historical launchers: public_source/prepare_legacy_exports.py,\npublic_source/demo_preparation_synthetic.py, and\ncheckpoint_lineage/deterministic_training_launcher.py. The listed ranker TXT file\nis an exact source excerpt; source comments are not newly verified score claims.\n\nLICENSE-SOURCE is the unmodified official Apache License 2.0 text:\nhttps://www.apache.org/licenses/LICENSE-2.0.txt\nSHA256: cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30\n'}
for p, s in FILES.items():
    os.makedirs(os.path.dirname(f'{SRC}/{p}'), exist_ok=True); open(f'{SRC}/{p}', 'w').write(s)
def get(cmd, dst):
    if not os.path.exists(dst + '/.ok'):
        os.makedirs(dst, exist_ok=True); sh(cmd + f' -p {dst}'); open(dst + '/.ok', 'w').close()
get('kaggle competitions download enveda-CASMI26-molecule-id-mass-spectra -f train.parquet', R + '/comp')
if glob.glob(R + '/comp/*.zip'): sh(f'cd {R}/comp && unzip -oq "*.zip" && rm -f *.zip')
get('kaggle datasets download prvsiyan/coconut-casmi26-candidates --unzip', R + '/coco')
get('kaggle datasets download dmitriigluzdov/casmi26-fold-safe-fpnet -f split.parquet', R + '/split')
if glob.glob(R + '/split/*.zip'): sh(f'cd {R}/split && unzip -oq "*.zip" && rm -f *.zip')
TRAIN = glob.glob(R + '/comp/**/train.parquet', recursive=True)[0]
COCO = os.path.dirname(glob.glob(R + '/coco/**/coco_fp.npy', recursive=True)[0])
SPLIT = glob.glob(R + '/split/**/split.parquet', recursive=True)[0]
print(TRAIN, COCO, SPLIT)
sh('nvidia-smi --query-gpu=name,memory.total --format=csv; free -g; nproc')


In [ ]:
# 2. MassSpecGym overlap with the withheld structures (a few minutes)
if not os.path.exists(f'{OUT}/msg_overlap.json'):
    sh(f'{sys.executable} {SRC}/msg_overlap.py {SPLIT} {OUT}/msg_overlap.json', check=False)


In [ ]:
# 3. trainer inputs (cached on Drive, so a reconnect skips this)
W = R + '/work'
need = ['ds/spec.npz', 'up_train/fp_meta.pkl', 'pool_fp.npy', 'pool_mass.npy', 'pool_meta.pkl']
if all(os.path.exists(f'{OUT}/prep/{f}') for f in need):
    for f in need:
        os.makedirs(os.path.dirname(f'{W}/{f}'), exist_ok=True)
        if not os.path.exists(f'{W}/{f}'): sh(f'cp {OUT}/prep/{f} {W}/{f}')
else:
    sh(f'{sys.executable} {SRC}/prep_e2.py {TRAIN} {COCO} {SPLIT} {W} --workers {os.cpu_count()} > {OUT}/prep.log 2>&1', check=False)
    print(open(f'{OUT}/prep.log').read()[-3000:])
    for f in need:
        os.makedirs(os.path.dirname(f'{OUT}/prep/{f}'), exist_ok=True); sh(f'cp {W}/{f} {OUT}/prep/{f}')


In [ ]:
# 4. train both models (original recipes; each skipped once done, resumed from its best checkpoint otherwise)
RUNS = [('fp_single_s2_f0', 'train_fp3.py', '--steps 24000 --seed 202 --max_minutes 300'),
        ('fp_merged_m1_f0', 'train_fp4.py', '--steps 50000 --seed 404 --merge_p 0.60 --max_minutes 420')]
for tag, trainer, extra in RUNS:
    if os.path.exists(f'{OUT}/{tag}.done'):
        print(tag, 'already done'); continue
    sh(f'cd {W} && PYTHONPATH={SRC}/third_party {sys.executable} -u {SRC}/third_party/{trainer} --spec ds/spec.npz --fpdir up_train '
       f'--pooldir . --bs 256 --lr 2e-4 --K 63 --lam 1.0 --d 512 --layers 6 --val_every 2000 {extra} '
       f'--out {OUT}/{tag}.pt --resume {OUT}/{tag}.pt >> {OUT}/{tag}.log 2>&1', check=False)
    log = open(f'{OUT}/{tag}.log').read()
    print(log[-1500:])
    if 'done. best hardneg-top1' in log: open(f'{OUT}/{tag}.done', 'w').close()


In [ ]:
# 5. summary
for tag, _, _ in RUNS:
    log = open(f'{OUT}/{tag}.log').read() if os.path.exists(f'{OUT}/{tag}.log') else ''
    vals = [l for l in log.splitlines() if 'VAL step' in l]
    print(tag, 'done' if os.path.exists(f'{OUT}/{tag}.done') else 'NOT done', '|', vals[-1] if vals else 'no validation yet')
sh(f'ls -la {OUT}')
